# Tree-Based Models

Unlike the previous notebook, this stage of the project is devided into two separate notebooks based on the preprocessing pipeline required by each model. RandomForest and XGBoost, both rely on the same preprocessing approach and are therefore evaluated together in this notebook. CatBoost, however, uses a different preprocessing strategy, by handling categorical features natively and is evaluated separately in the next notebook.

Raandom Forest serves as the baseline tree-based model for both notebooks. Its evaluated metrics are saved to a CSV file so they can be reused in the CatBoost notebook, allowing all tree-based models to be compared consistently without retraining the baseline model.

In [1]:
%load_ext autoreload
%autoreload 2

from time import perf_counter

NOTEBOOK_START = perf_counter()

import pandas as pd
import numpy as np
import sys
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 500)

sys.path.append(str(Path().cwd().parent.resolve()))

import preprocessing.features as features

builder = features.FeatureBuilder()

df = pd.read_csv(features.DATASET_PATH)

df_copy = builder.get_df(df)

df_copy.shape

/home/carl/notebooks/airbnb_prices_prediction/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(74111, 27)

In [2]:
import xgboost

xgboost.__version__

'3.3.0'

In [3]:
import sklearn

sklearn.__version__

'1.9.0'

# Baseline Model. RandomForest

In [4]:
df_copy = builder.get_df(df, use_amenities=False, use_embeddings=False)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

((74111, 25), (74111,))

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape

((59288, 25), (14823, 25))

In [6]:
import preprocessing.tree_preprocessor as preprocessor
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

cat_features = X_train.select_dtypes(include=['string', 'object']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

tree_pipeline = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

tree_pipeline.fit(X_train, y_train)

y_pred_test_log = tree_pipeline.predict(X_test)
y_pred_train_log = tree_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 52.22$ | Train MAE: 21.08$
Test RMSE: 118.45$ | Train RMSE: 56.19$
Test R2 Score: 0.68 | Train R2 Score: 0.95


## RandomForest Conclusion

The baseline RandomForest model exhibit significant overfitting, with substantially better performance on the training set that on the test set. Therefore, its current evaluation metrics are not suitable as the primary baseline for comparing tree-based models. In the next step, hyperparameter tuning will be performed using RandomizedSearchCV to reduce overfitting and establish a more reliable baseline for subsequent comparisons. 

# RandomForest. Hyperparameter Tuning.

In [7]:
RANDOM_FOREST_DIR = Path('../artifacts/random_forest')
RANDOM_FOREST_DIR.mkdir(exist_ok=True, parents=True)

BASELINE_RANDOM_SEARCH_MODEL = RANDOM_FOREST_DIR / "random_forest_random_search.joblib"

In [8]:
%%time

import joblib
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

if BASELINE_RANDOM_SEARCH_MODEL.exists():
    print("Loading RandomizedSearchCV...")
    random_search = joblib.load(BASELINE_RANDOM_SEARCH_MODEL)
else:
    print("Training RandomizedSearchCV...")
    
    tree_pipeline = Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", RandomForestRegressor(random_state=42))
    ])
    
    param_dist = {
        "model__n_estimators": randint(150, 600),
        "model__max_depth": [10, 15, 20, 25, 30],
        "model__min_samples_split": randint(10, 40),
        "model__min_samples_leaf": randint(3, 15),
        "model__max_features": ['log2', 'sqrt', 0.3, 0.3],
        "model__ccp_alpha": [0.0, 0.0001, 0.0005, 0.001]
    }
    
    random_search = RandomizedSearchCV(
        estimator=tree_pipeline,
        param_distributions=param_dist,
        n_iter=15,
        scoring="neg_root_mean_squared_error",
        cv=3,
        random_state=42,
        n_jobs=-1,
        verbose=2
    )

    random_search.fit(X_train, y_train)
    joblib.dump(random_search, BASELINE_RANDOM_SEARCH_MODEL)

best_rf = random_search.best_estimator_
random_search.best_params_

Loading RandomizedSearchCV...
CPU times: user 84.3 ms, sys: 50.2 ms, total: 135 ms
Wall time: 145 ms


{'model__ccp_alpha': 0.0,
 'model__max_depth': 25,
 'model__max_features': 0.3,
 'model__min_samples_leaf': 10,
 'model__min_samples_split': 12,
 'model__n_estimators': 299}

In [9]:
y_pred_test_log = best_rf.predict(X_test)
y_pred_train_log = best_rf.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 53.55$ | Train MAE: 46.96$
Test RMSE: 121.80$ | Train RMSE: 108.13$
Test R2 Score: 0.67 | Train R2 Score: 0.74


In [10]:
results_df = pd.DataFrame(
    columns=['MAE', 'RMSE', 'R2']
)

results_df.loc['RandomForest (baseline, optimized)'] = [
    round(test_MAE, 2),
    round(test_RMSE, 2),
    round(test_r2, 2)
]

results_df

,MAE,RMSE,R2
"RandomForest (baseline, optimized)",53.55,121.8,0.67


## RandomForest. Hypyerparameter tuning Conclusion.

The primary objective of hyperparameter tuning was not to maximize predictive performance, but to reduce overfitting and obtain a more stable baseline model for comparison with other algorithms. This objective was sucessfully achieved by shifting the hyperparameter search toward stronger regularization, which significantly reduced the gap between the training and test performance. Although a moderate degree of overfitting still remains, it is acceptable for the purposes of this project and provides a reliable baseline for evaluating more advanced tree-based methods.

To avoid retraining during subsequent executions of the notebook, the optimized model was saved in the `./artifacts/` directory and can be loaded directly when needed. This reduce both computational cost and notebook execution time, while ensuring reproducible results.

# XGBoost

In this section, the **native XGBoost API** is used instead of the **scikit-learn** wrapper, providing greater flexibility over the training process. The approach enables the use of **GPU acceleration**, **Early Stopping** and seamless integration with **Optuna** by hyperparameter optimization.

Hyperparameter tuning is performed using **Optuna**, which employs an efficient search strategy to identify high-performing parameter configurations while requiring significantly fewer model evaluations than an exhaustive **Grid Search**. This allows the optimization process to be completed substantially faster without compromising model quality.

To ensure a fair comparison across all XGBoost experiments, the same optimization procedure, hyperparameter search space and training configuration are used for every model. The only difference between experiments is the set of input features (baseline, `zipcode`, `amenities`, `description embeddings` and thier combinations), allowing the impact of each feature set on the final predictive performance to be evaluated independently.

## XGBoost Baseline

The baseline XGBoost model is trained using the original feature set without `amenities`, `description embeddings` or `zipcode`. The model serves as a reference point for evaluating contribution of additional features in a subsequent experiments.

In [11]:
df_copy = builder.get_df(df, use_amenities=False, use_embeddings=False)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

((74111, 25), (74111,))

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape, X_val.shape

((47430, 25), (14823, 25), (11858, 25))

In [13]:
cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

X_train_processed = tree_preprocessor.fit_transform(X_train, y_train)

X_val_processed = tree_preprocessor.transform(X_val)
X_test_processed = tree_preprocessor.transform(X_test)

X_train_processed = X_train_processed.astype(np.float32)
X_val_processed = X_val_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

In [14]:
import xgboost as xgb
DEVICE = "cuda" if xgb.build_info()['USE_CUDA'] else "cpu"

if DEVICE == 'cuda':
    dtrain = xgb.QuantileDMatrix(X_train_processed, label=y_train)
    dval = xgb.QuantileDMatrix(X_val_processed, label=y_val, ref=dtrain)
    dtest = xgb.QuantileDMatrix(X_test_processed, label=y_test, ref=dtrain)
else:
    dtrain = xgb.DMatrix(X_train_processed, label=y_train)
    dval = xgb.DMatrix(X_val_processed, label=y_val)
    dtest = xgb.DMatrix(X_test_processed, label=y_test)

In [15]:
from optuna_integration import XGBoostPruningCallback
import xgboost as xgb

print(f"Using: {DEVICE} for objective funcion.")

def objective(trial):

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "device": DEVICE,
        
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.8),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 20),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1, 20, log=True),

        "verbosity": 0,
        "n_jobs": -1,
        "seed": 42
    }

    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=1000,
        evals=[(dval, "validation")],
        early_stopping_rounds=50,
        verbose_eval=False,
        callbacks=[XGBoostPruningCallback(trial, "validation-rmse")]
    )
    
    predictions = booster.predict(dval)
    rmse = root_mean_squared_error(y_val, predictions)
    
    return rmse

Using: cuda for objective funcion.


In [16]:
XGBOOST_DIR = Path('../artifacts/xgboost')
XGBOOST_DIR.mkdir(exist_ok=True, parents=True)

XGBOOST_BASELINE = XGBOOST_DIR / "xgboost_baseline"

In [17]:
%%time

import optuna
from utils.xgb_pipeline import XGBoostPipeline

if (
    XGBOOST_BASELINE.with_suffix(".json").exists() and
    XGBOOST_BASELINE.with_suffix(".joblib").exists() and
    XGBOOST_BASELINE.with_suffix(".params").exists()
):
    print("Loading XGboost Baseline model...")
    xgboost_pipeline = XGBoostPipeline.load(XGBOOST_BASELINE)
else:
    print("Training XGBoost Baseline model...")
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=50, interval_steps=10)
    )
    
    study.optimize(objective, n_trials=250, gc_after_trial=True)

    best_params = {
        **study.best_params,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "n_jobs": -1,
        "seed": 42,
        "verbosity": 0,
        "device": DEVICE
    }

    xgboost_pipeline = XGBoostPipeline(preprocessor=tree_preprocessor)
    xgboost_pipeline.fit(X_train, y_train, params=best_params, X_val=X_val, y_val=y_val)

    xgboost_pipeline.save(XGBOOST_BASELINE)

xgboost_pipeline.params

Loading XGboost Baseline model...
CPU times: user 83.5 ms, sys: 1.01 ms, total: 84.5 ms
Wall time: 45.5 ms


{'learning_rate': 0.07932485635929769,
 'max_depth': 6,
 'subsample': 0.7513721112350875,
 'colsample_bytree': 0.5236549417781432,
 'gamma': 0.16064811158072703,
 'min_child_weight': 15,
 'reg_alpha': 0.8926771612494401,
 'reg_lambda': 3.6674474492877636,
 'objective': 'reg:squarederror',
 'eval_metric': 'rmse',
 'tree_method': 'hist',
 'n_jobs': -1,
 'seed': 42,
 'verbosity': 0,
 'device': 'cuda'}

In [18]:
y_pred_test_log = xgboost_pipeline.predict(X_test)
y_pred_train_log = xgboost_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 49.91$ | Train MAE: 42.79$
Test RMSE: 113.39$ | Train RMSE: 96.63$
Test R2 Score: 0.71 | Train R2 Score: 0.78


In [19]:
results_df.loc['XGBoost (baseline, optimized)'] = (
    round(test_MAE, 2),
    round(test_RMSE, 2),
    round(test_r2, 2)
)

results_df

,MAE,RMSE,R2
"RandomForest (baseline, optimized)",53.55,121.80,0.67
"XGBoost (baseline, optimized)",49.91,113.39,0.71


### Conclusion

The baseline XGBoost model outperformed a baseline RandomForest baseline model across all evaluation metrics. Compared to the RandomForest model, XGBosot reduced prediction error by approximately **$3.9 MAE** and improved coefficient of determination from **0.67** to **0.71**.

The difference between training and test metric remains relatively small, indicating only moderate overfitting and demonstrating good generalization on unseen data.

These results establish XGBoost as a stronger baseline model for the remaining experiments. Subsequent sections investigate whether incorporating additional information such as `zipcode`, `amenities` and `description embeddings` can further improve prediction performance.

## XGBoost Amenities + Description Embeddings

In [20]:
df_copy = builder.get_df(df, use_amenities=True, use_embeddings=True)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

Loading embeddings from Parquet /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings.parquet...


((74111, 1166), (74111,))

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_val.shape, X_test.shape

((47430, 1166), (11858, 1166), (14823, 1166))

In [22]:
import xgboost as xgb

cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

X_train_processed = tree_preprocessor.fit_transform(X_train, y_train)
X_val_processed = tree_preprocessor.transform(X_val)
X_test_processed = tree_preprocessor.transform(X_test)

X_train_processed = X_train_processed.astype(np.float32)
X_val_processed = X_val_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

if DEVICE == "cuda":
    dtrain = xgb.QuantileDMatrix(X_train_processed, label=y_train)
    dval = xgb.QuantileDMatrix(X_val_processed, label=y_val, ref=dtrain)
    dtest = xgb.QuantileDMatrix(X_test_processed, label=y_test, ref=dtrain)
else:
    dtrain = xgb.DMatrix(X_train_processed, label=y_train)
    dval = xgb.DMatrix(X_val_processed, label=y_val)
    dtest = xgb.DMatrix(X_test_processed, label=y_test)

In [23]:
XGBOOST_DIR = Path('../artifacts/xgboost')
XGBOOST_DIR.mkdir(exist_ok=True, parents=True)

XGBOOST_AMENITIES_EMBEDDINGS = XGBOOST_DIR / "xgboost_amenities_embeddings"

In [24]:
%%time

from utils.xgb_pipeline import XGBoostPipeline


if (
    XGBOOST_AMENITIES_EMBEDDINGS.with_suffix(".json").exists() and
    XGBOOST_AMENITIES_EMBEDDINGS.with_suffix(".joblib").exists() and
    XGBOOST_AMENITIES_EMBEDDINGS.with_suffix(".params").exists()
):
    print("Loading XGboost (Full dataset) Pipeline...")
    xgboost_pipeline = XGBoostPipeline.load(XGBOOST_AMENITIES_EMBEDDINGS)
else:
    print("Training XGBoost (Full dataset) Pipeline...")

    DEVICE = "cuda" if xgb.build_info()['USE_CUDA'] else "cpu"
    print(f"Training model, using device: {DEVICE}")
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=50, interval_steps=10)
    )
    
    study.optimize(objective, n_trials=250, gc_after_trial=True)

    best_params = {
        **study.best_params,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "n_jobs": -1,
        "seed": 42,
        "verbosity": 0,
        "device": DEVICE
    }

    xgboost_pipeline = XGBoostPipeline(preprocessor=tree_preprocessor)
    xgboost_pipeline.fit(
        X_train, y_train, params=best_params, X_val=X_val, y_val=y_val
    )

    xgboost_pipeline.save(XGBOOST_AMENITIES_EMBEDDINGS)

xgboost_pipeline.params

Loading XGboost (Full dataset) Pipeline...
CPU times: user 101 ms, sys: 0 ns, total: 101 ms
Wall time: 58.7 ms


{'learning_rate': 0.055557374131755995,
 'max_depth': 6,
 'subsample': 0.8327022046739874,
 'colsample_bytree': 0.7836736730919611,
 'gamma': 0.09674908315129453,
 'min_child_weight': 13,
 'reg_alpha': 0.06930070064357938,
 'reg_lambda': 1.3746422487560193,
 'objective': 'reg:squarederror',
 'eval_metric': 'rmse',
 'tree_method': 'hist',
 'n_jobs': -1,
 'seed': 42,
 'verbosity': 0,
 'device': 'cuda'}

In [25]:
y_pred_test_log = xgboost_pipeline.predict(X_test)
y_pred_train_log = xgboost_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 48.68$ | Train MAE: 28.52$
Test RMSE: 111.48$ | Train RMSE: 64.03$
Test R2 Score: 0.72 | Train R2 Score: 0.91


In [26]:
results_df.loc['XGBoost (amenities+embeddings, optimized)'] = (
    round(test_MAE, 2),
    round(test_RMSE, 2),
    round(test_r2, 2)
)

results_df

,MAE,RMSE,R2
"RandomForest (baseline, optimized)",53.55,121.80,0.67
"XGBoost (baseline, optimized)",49.91,113.39,0.71
"XGBoost (amenities+embeddings, optimized)",48.68,111.48,0.72


### Conclusion

The inclusion of `amenities` and `description embeddings` resulted in a modest improvement across the evaluation metrics. However, these additional features substantially increased the dimensionality of the dataset, leading to a training time that was approximately **2-4 times longer** than that of the baseline model.

Furthermore, the high dimensional embeddings representation (1024 features) was accompined by noticeably higher degree of overfitting, as reflected by a larger gap between training and test performance. While the model achieved slightly better predictive accuracy, the improvement was relatively small, compared to the additional computational cost and increased model complexity.

Overall, this experiment demonstrates that the limited performance gain does not justify the significantly longer training time and the increased overfitting, making the baseline XGBoost a more practical choice in terms of trade-off between predictive performance and computational efficiency.

## PCA For Description Embeddings

Before evaluating the impact of PCA on the XGBoost model, we first formulate a hypothesis that dimensionality reduction of description embeddings can improve overall training process without sacrificing prediction performance. The experiment is motivated by three primary objectives:

* Reduce training time by decreasing the number of description embeddings;
* Reduce overfitting by removing redundant and noisy dimensions;
* Preserve predictive performance, ensuring that dimensionality reduction does not degrade the model's accuracy;

To determine an appropriate number of principal components, we first analyze cumulative explained variance of the embedding space. Based on this analysis, several PCA configurations are evaluated and compared with the original 1024-dimensional embeddings to identify the best trade-off between dimensionality reduction, model generalization and predictive performance.

In [27]:
embeddings_df = builder.get_description_embeddings_df(df)

embeddings_df.shape

Loading embeddings from Parquet /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings.parquet...


(74111, 1024)

In [28]:
from sklearn.decomposition import PCA

pca = PCA()
pca.fit(embeddings_df)

explained_variance = np.cumsum(pca.explained_variance_ratio_)

explained_variance[:10]

array([0.04633991, 0.07908731, 0.10364504, 0.12456712, 0.14230436,
       0.15870565, 0.17409456, 0.18907511, 0.20272037, 0.21572368],
      dtype=float32)

In [29]:
for threshold in [0.90, 0.95, 0.96, 0.97, 0.99]:
    n_components = np.argmax(explained_variance >= threshold) + 1
    print(f"{threshold:.0%} variance: {n_components} components")

90% variance: 278 components
95% variance: 370 components
96% variance: 399 components
97% variance: 438 components
99% variance: 605 components


### N_Components 370

The first experiment uses **370 principal components**, which preserves approximately 95% of the total variance in the original 1024-dimensional embedding space. This threshold is commonly used in dimensionality reduction as it retains the vast majority of the information while substantially reducing the number of features. For this reason, it serves as a natural starting point for evaluating the effect of PCA on model performance and generalization.

In [30]:
pca = PCA(n_components=370, random_state=42)

embeddings_pca = pca.fit_transform(embeddings_df)

embeddings_pca.shape

(74111, 370)

In [31]:
embeddings_pca = pd.DataFrame(
    embeddings_pca,
    index=df_copy.index,
    columns=[f"embedding_pca_{i}" for i in range(embeddings_pca.shape[1])]
)

df_copy = builder.get_df(df, use_amenities=True, use_embeddings=False)

df_copy = pd.concat([df_copy, embeddings_pca], axis=1)

df_copy.shape

(74111, 514)

In [32]:
X = df_copy.drop(columns='log_price')
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

((74111, 512), (74111,))

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape, X_val.shape

((47430, 512), (14823, 512), (11858, 512))

In [34]:
cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

params = xgboost_pipeline.params

pca_pipeline = XGBoostPipeline(
    preprocessor=tree_preprocessor
)

pca_pipeline.fit(X_train, y_train, params=params, X_val=X_val, y_val=y_val)

y_pred_test_log = pca_pipeline.predict(X_test)
y_pred_train_log = pca_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 48.14$ | Train MAE: 27.29$
Test RMSE: 110.46$ | Train RMSE: 61.62$
Test R2 Score: 0.73 | Train R2 Score: 0.91


The original model with 1024-dimensional embedding space:

Test MAE: 48.62 | Train MAE: 24.59

Test RMSE: 111.60 | Train RMSE: 55.64

Test R2 Score: 0.73 | Train R2 Score: 0.93

#### Conclusion 

Applying PCA with **370 principal components**, produced encouraging results. The gap between training and test metric became noticeably smaller, indicating a reduction in overfitting compared to the original 1024-dimensional embeddings. At the same time, the model preserved its predictive performance, with the test metrics showing a slight improvement over the original model.

This results suggest that a significant portion of the embedding dimensions is redundant for the XGBoost model. As a result, the next experiment investigates a more aggresive dimensionality reduction of **278 principal components**, which preserve **90% of the total explained variance**, to determine whether further reducing the embedding space can improve generalization without sacrificing predictive performance.  

### N_Components 278

In [35]:
embeddings_df = builder.get_description_embeddings_df(df)

pca = PCA(n_components=278, random_state=42)

embeddings_pca = pca.fit_transform(embeddings_df)

embeddings_pca.shape

Loading embeddings from Parquet /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings.parquet...


(74111, 278)

In [36]:
embeddings_pca = pd.DataFrame(
    embeddings_pca,
    index=df_copy.index,
    columns=[f"embedding_pca_{i}" for i in range(embeddings_pca.shape[1])]
)

df_copy = builder.get_df(df, use_amenities=True, use_embeddings=False)

df_copy = pd.concat([df_copy, embeddings_pca], axis=1)

df_copy.shape

(74111, 422)

In [37]:
X = df_copy.drop(columns='log_price')
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

((74111, 420), (74111,))

In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape, X_val.shape

((47430, 420), (14823, 420), (11858, 420))

In [39]:
cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

params = xgboost_pipeline.params

pca_pipeline = XGBoostPipeline(
    preprocessor=tree_preprocessor
)

pca_pipeline.fit(X_train, y_train, params=params, X_val=X_val, y_val=y_val)

y_pred_test_log = pca_pipeline.predict(X_test)
y_pred_train_log = pca_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 48.24$ | Train MAE: 28.14$
Test RMSE: 110.63$ | Train RMSE: 63.83$
Test R2 Score: 0.73 | Train R2 Score: 0.91


The original model with 1024-dimensional embedding space:

Test MAE: 48.62 | Train MAE: 24.59

Test RMSE: 111.60 | Train RMSE: 55.64

Test R2 Score: 0.73 | Train R2 Score: 0.93

#### Conclusion 

Using **278 principal components** further reduced the embedding dimensionality while preserving **90% of the total explained variance**. Compared to the originall 1024-dimensional embeddings, the gap between the training and test metrics remained noticeably smaller, confirming that PCA continued to navigate overfitting. However, the reduction was in overfitting was not as pronounced as in the previous experiment with 370 components.

Intrestingly, the configuration achieved the best predictive performance among all evaluated embeddings representations, slightly outperforming both the original 1024-dimensional embeddings and the 370-component PCA model. However, the primiary objective of this study was not to maximize the predictive performance, but to identify a dimensionality reduction strategy that effectively reduces overfitting while preserving model quality. From this perspective, the **370-component configuration** provides a better balance between generalization and predictive performance, making it the perfect choice for subsequent experiments.

## PCA For Description Embeddings. Conclusion

Based on the conducted experiments, `n_components=370` **was selected as the best trade-off** between dimensionality reduction, model generalization and preservation of predictive performance.

To improve reusability and flexibility of the feature engineering pipeline, the `FeatureBuilder()` class was updated (`./preprocessing/features.py`). A new parameter, `embedding_pca_components` was intoduced, allowing optional PCA transformation for description embeddings during feature generation.

The default value is set to `None`, meaning that the original embeddings will be used unless dimensionality reduction explicitly requested.

Example usage:

    builder = FeatureBuilder()
    df_copy = builder.get_df(df, use_embeddings=True, embedding_pca_components=370)

This approach allows the same feature engineering pipeline to support both original embeddings and PCA-reduced representations without requiring additional preprocessing logic.

## XGBoost Baseline + Zipcode

In [40]:
df_copy = builder.get_df(df, use_amenities=False, use_embeddings=False)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X.shape, y.shape

((74111, 26), (74111,))

In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_val.shape, X_test.shape

((47430, 26), (11858, 26), (14823, 26))

In [42]:
cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

X_train_processed = tree_preprocessor.fit_transform(X_train, y_train)
X_val_processed = tree_preprocessor.transform(X_val)
X_test_processed = tree_preprocessor.transform(X_test)

X_train_processed = X_train_processed.astype(np.float32)
X_val_processed = X_val_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

DEVICE = 'cuda' if xgb.build_info()['USE_CUDA'] else 'cpu'
print(f"Currently using device is: {DEVICE}")

if DEVICE == 'cuda':
    dtrain = xgb.QuantileDMatrix(X_train_processed, label=y_train)
    dval = xgb.QuantileDMatrix(X_val_processed, label=y_val, ref=dtrain)
    dtest = xgb.QuantileDMatrix(X_test_processed, label=y_test, ref=dtrain)
else:
    dtrain = xgb.DMatrix(X_train_processed, label=y_train)
    dval = xgb.DMatrix(X_val_processed, label=y_val)
    dtest = xgb.DMatrix(X_test_processed, label=y_test)

Currently using device is: cuda


In [43]:
XGBOOST_DIR = Path('../artifacts/xgboost')
XGBOOST_DIR.mkdir(exist_ok=True, parents=True)

XGBOOST_BASELINE_ZIPCODE = XGBOOST_DIR / "xgboost_baseline_zipcode"

In [44]:
%%time

if (
    XGBOOST_BASELINE_ZIPCODE.with_suffix(".json").exists() and
    XGBOOST_BASELINE_ZIPCODE.with_suffix(".joblib").exists() and 
    XGBOOST_BASELINE_ZIPCODE.with_suffix(".params").exists()
):
    print("Loading XGBoost Baseline Model with zipcode...")
    xgboost_pipeline = XGBoostPipeline.load(XGBOOST_BASELINE_ZIPCODE)
else:
    print("Training XGBoost Baseline Model with zipcode...")
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=50, interval_steps=10)
    )

    study.optimize(objective, n_trials=250, gc_after_trial=True)

    best_params = {
        **study.best_params,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "n_jobs": -1,
        "seed": 42,
        "verbosity": 0,
        "device": DEVICE
    }
    xgboost_pipeline = XGBoostPipeline(preprocessor=tree_preprocessor)
    xgboost_pipeline.fit(
        X_train, y_train,
        params=best_params,
        X_val=X_val, y_val=y_val
    )

    xgboost_pipeline.save(XGBOOST_BASELINE_ZIPCODE)

xgboost_pipeline.params

Loading XGBoost Baseline Model with zipcode...
CPU times: user 79.2 ms, sys: 1.02 ms, total: 80.2 ms
Wall time: 40.3 ms


{'learning_rate': 0.07357441328174337,
 'max_depth': 6,
 'subsample': 0.7482036765964312,
 'colsample_bytree': 0.5558864498315155,
 'gamma': 0.22725637524524783,
 'min_child_weight': 15,
 'reg_alpha': 0.0031959066220104107,
 'reg_lambda': 1.204409424885996,
 'objective': 'reg:squarederror',
 'eval_metric': 'rmse',
 'tree_method': 'hist',
 'n_jobs': -1,
 'seed': 42,
 'verbosity': 0,
 'device': 'cuda'}

In [45]:
y_pred_test_log = xgboost_pipeline.predict(X_test)
y_pred_train_log = xgboost_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 49.35$ | Train MAE: 42.99$
Test RMSE: 112.69$ | Train RMSE: 97.20$
Test R2 Score: 0.72 | Train R2 Score: 0.77


In [46]:
results_df.loc['XGBoost (baseline, optimized, zipcode)'] = (
    round(test_MAE, 2),
    round(test_RMSE, 2),
    round(test_r2, 2)
)

results_df.sort_values(by='RMSE', ascending=False)

,MAE,RMSE,R2
"RandomForest (baseline, optimized)",53.55,121.80,0.67
"XGBoost (baseline, optimized)",49.91,113.39,0.71
"XGBoost (baseline, optimized, zipcode)",49.35,112.69,0.72
"XGBoost (amenities+embeddings, optimized)",48.68,111.48,0.72


### Conclusion

Adding the **zipcode** feature resulted in only a marginal change in the model's performance. While R2 Score increased slightly, the overall predictive performance remained the nearly identical to the baseline model, with no meaningful improvement in the primary evaluation metric (RMSE). The training and test metrics also remained as a similar level, indicating that the additional information did not noticeably affect the model's generalization.

Overall, the **zipcode** feature provides only a limited contribution when used with the baseline feature set, suggesting that the existing numerical and categorical features already capture most of the available location-related information.

## XGBoost Amenities, Description Embeddings + Zipcode

In [47]:
df_copy = builder.get_df(df, use_amenities=True, use_embeddings=True, embedding_pca_components=370)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X.shape, y.shape

Loading embeddings from Parquet /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings.parquet...
Applying PCA (370 components)...
Embeddings shape is: (74111, 370)


((74111, 513), (74111,))

In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_val.shape, X_test.shape

((47430, 513), (11858, 513), (14823, 513))

In [49]:
cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

X_train_processed = tree_preprocessor.fit_transform(X_train, y_train)
X_val_processed = tree_preprocessor.transform(X_val)
X_test_processed = tree_preprocessor.transform(X_test)

X_train_processed = X_train_processed.astype(np.float32)
X_val_processed = X_val_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

DEVICE = 'cuda' if xgb.build_info()['USE_CUDA'] else 'cpu'
print(f"Currently using device is: {DEVICE}")

if DEVICE == 'cuda':
    dtrain = xgb.QuantileDMatrix(X_train_processed, label=y_train)
    dval = xgb.QuantileDMatrix(X_val_processed, label=y_val, ref=dtrain)
    dtest = xgb.QuantileDMatrix(X_test_processed, label=y_test, ref=dtrain)
else:
    dtrain = xgb.DMatrix(X_train_processed, label=y_train)
    dval = xgb.DMatrix(X_val_processed, label=y_val)
    dtest = xgb.DMatrix(X_test_processed, label=y_test)

Currently using device is: cuda


In [50]:
XGBOOST_DIR = Path('../artifacts/xgboost')
XGBOOST_DIR.mkdir(exist_ok=True, parents=True)

XGBOOST_AM_EM_ZIPCODE = XGBOOST_DIR / "xgboost_am_em_zipcode"

In [51]:
%%time

if (
    XGBOOST_AM_EM_ZIPCODE.with_suffix(".json").exists() and
    XGBOOST_AM_EM_ZIPCODE.with_suffix(".joblib").exists() and
    XGBOOST_AM_EM_ZIPCODE.with_suffix(".params").exists()
):
    print("Loading XGBoost Model with amenities, embeddings and zipcode...")
    xgboost_pipeline = XGBoostPipeline.load(XGBOOST_AM_EM_ZIPCODE)
else:
    print("Training XGBoost Model with amenities, embeddings and zipcode...")
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=50, interval_steps=10)
    )

    study.optimize(objective, n_trials=250, gc_after_trial=True)

    best_params = {
        **study.best_params,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "n_jobs": -1,
        "seed": 42,
        "verbosity": 0,
        "device": DEVICE
    }

    xgboost_pipeline = XGBoostPipeline(preprocessor=tree_preprocessor)
    xgboost_pipeline.fit(
        X_train, y_train,
        params=params,
        X_val=X_val, y_val=y_val
    )

    xgboost_pipeline.save(XGBOOST_AM_EM_ZIPCODE)

xgboost_pipeline.params

Loading XGBoost Model with amenities, embeddings and zipcode...
CPU times: user 96.7 ms, sys: 998 μs, total: 97.7 ms
Wall time: 56.4 ms


{'learning_rate': 0.055557374131755995,
 'max_depth': 6,
 'subsample': 0.8327022046739874,
 'colsample_bytree': 0.7836736730919611,
 'gamma': 0.09674908315129453,
 'min_child_weight': 13,
 'reg_alpha': 0.06930070064357938,
 'reg_lambda': 1.3746422487560193,
 'objective': 'reg:squarederror',
 'eval_metric': 'rmse',
 'tree_method': 'hist',
 'n_jobs': -1,
 'seed': 42,
 'verbosity': 0,
 'device': 'cuda'}

In [52]:
y_pred_test_log = xgboost_pipeline.predict(X_test)
y_pred_train_log = xgboost_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 48.33$ | Train MAE: 30.45$
Test RMSE: 111.21$ | Train RMSE: 68.55$
Test R2 Score: 0.73 | Train R2 Score: 0.89


In [53]:
results_df.loc['XGBoost (amenities, embeddings, optimized, zipcode, pca_components=370)'] = (
    round(test_MAE, 2),
    round(test_RMSE, 2),
    round(test_r2, 2)
)

results_df.sort_values(by='RMSE', ascending=False)

,MAE,RMSE,R2
"RandomForest (baseline, optimized)",53.55,121.80,0.67
"XGBoost (baseline, optimized)",49.91,113.39,0.71
"XGBoost (baseline, optimized, zipcode)",49.35,112.69,0.72
"XGBoost (amenities+embeddings, optimized)",48.68,111.48,0.72
"XGBoost (amenities, embeddings, optimized, zipcode, pca_components=370)",48.33,111.21,0.73


### Conclusion

Adding the `zipcode` feature to the full feature set resulted in a small improvement in predictive performance, producing the best overall test metrics among the evaluated XGBoost models. However, the improvement came at the cost of increased overfitting, as the gap between the training and test metrics became noticeably larger compared to the PCA-based model without the zipcode feature. 

These results suggest that while **zipcode** provides additional location-specific information, that benefits prediction accuracy, it also increases the model's tendency to memorize the training data. Overall, the improvement in predictive performance is relatively modest, making the trade-off between accuracy and generalization an important consideration.

# Overfitting Reduction

## Cross-Validation Reference Model

Previous XGBoost experiments, consistently exhibited a noticeable degree of overfitting. As a initial step adressing this issue, a new **reference model** will be trained using `xgb.cv()` instead of a single validation split. This experiment aims to determine whether relying on a single validation set, rather than multiple cross-validation folds, contributes to the observed overfitting.

The reference model is built using the best configuration identified in the previous chapters. Specifically, it uses **370-PCA** components extracted from description embeddings (`N_COMPONENTS=370`), excludes the `zipcode` feature and performs hyperparameter optimization with *Optuna* for **250 trials**.

Regardless of whether cross-validation alone reduces overfitting, this model serves as a new reference point for all subsequent experiments. At the end of this section, its performance will be compared with the previously most stable **XGBoost with amenities, description embeddings** to quantify the impact of replacing the hold-out validation strategy with cross-validation.

In [54]:
def objective(trial):

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "device": DEVICE,
        
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.8),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 20),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1, 20, log=True),

        "verbosity": 0,
        "n_jobs": -1,
        "seed": 42
    }

    cv_results = xgb.cv(
        params=params,
        dtrain=dtrain,
        early_stopping_rounds=50,
        num_boost_round=1000,
        nfold=3,
        metrics="rmse",
        seed=42,
        verbose_eval=False,
        shuffle=True,
        callbacks=[XGBoostPruningCallback(trial, "test-rmse")]
    )

    trial.set_user_attr(
        "best_num_boost_round",
        len(cv_results)
    )

    mean_rmse = cv_results['test-rmse-mean'].values[-1]

    return mean_rmse

In [55]:
EMBEDDING_PCA_COMPONENTS = 370

df_copy = builder.get_df(df, use_amenities=True, use_embeddings=True, embedding_pca_components=EMBEDDING_PCA_COMPONENTS)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

Loading embeddings from Parquet /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings.parquet...
Applying PCA (370 components)...
Embeddings shape is: (74111, 370)


((74111, 512), (74111,))

In [56]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape

((59288, 512), (14823, 512))

In [57]:
cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

X_train_processed = tree_preprocessor.fit_transform(X_train, y_train)
X_test_processed = tree_preprocessor.transform(X_test)

X_train_processed = X_train_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

dtrain = xgb.DMatrix(X_train_processed, label=y_train)
dtest = xgb.DMatrix(X_test_processed, label=y_test)

In [58]:
XGBOOST_DIR = Path('../artifacts/xgboost')
XGBOOST_DIR.mkdir(exist_ok=True)

XGBOOST_REFERENCE = XGBOOST_DIR / "xgboost_reference"

In [59]:
%%time

if (
    XGBOOST_REFERENCE.with_suffix(".json").exists() and
    XGBOOST_REFERENCE.with_suffix(".joblib").exists() and
    XGBOOST_REFERENCE.with_suffix(".params").exists()
):
    print("Loading Reference XGBoost model...")
    xgboost_pipeline = XGBoostPipeline.load(XGBOOST_REFERENCE)
else:
    print("Training Reference XGBoost model...")
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=50, interval_steps=10)
    )

    study.optimize(objective, n_trials=50, gc_after_trial=True)
    best_num_boost_round = study.best_trial.user_attrs['best_num_boost_round']

    best_params = {
        **study.best_params,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "n_jobs": -1,
        "seed": 42,
        "verbosity": 0,
        "device": DEVICE
    }

    xgboost_pipeline = XGBoostPipeline(preprocessor=tree_preprocessor)
    xgboost_pipeline.fit(
        X_train, y_train,
        params=best_params,
        num_boost_round=best_num_boost_round
    )

    xgboost_pipeline.save(XGBOOST_REFERENCE)

xgboost_pipeline.params

Loading Reference XGBoost model...
CPU times: user 105 ms, sys: 32 μs, total: 105 ms
Wall time: 65.8 ms


{'learning_rate': 0.04581977444178674,
 'max_depth': 6,
 'subsample': 0.6696446471492034,
 'colsample_bytree': 0.7108577778824173,
 'gamma': 0.32008556211228506,
 'min_child_weight': 16,
 'reg_alpha': 0.004944351124962353,
 'reg_lambda': 1.6608693305809532,
 'objective': 'reg:squarederror',
 'eval_metric': 'rmse',
 'tree_method': 'hist',
 'n_jobs': -1,
 'seed': 42,
 'verbosity': 0,
 'device': 'cuda'}

In [60]:
y_pred_test_log = xgboost_pipeline.predict(X_test)
y_pred_train_log = xgboost_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 48.19$ | Train MAE: 32.97$
Test RMSE: 110.35$ | Train RMSE: 74.28$
Test R2 Score: 0.73 | Train R2 Score: 0.88


**XGBoost model with amenities and description embeddings (without PCA)**:

Test MAE: 48.62 | Train MAE: 24.59

Test RMSE: 111.60 | Train RMSE: 55.64

Test R2 Score: 0.73 | Train R2 Score: 0.93

### Cross-Validation Conclusion

Replacing a single validation split with `xgb.cv()` demonstrated a clear tendency to reduce overfitting. The gap between the training and validation performance became noticeably smaller, indicating that cross-validation provides a more reliable estimate of the model's generalization ability during hyperparameter optimization.

However, the model still exhibits a considerable degree of overfitting, suggesting that the validation strategy alone is not the primary cause of the problem. A comparison with the previously most stable model, **XGBoost with amenities and description embeddings**, shows only a modest improvement in predictive performance. Therefore, while cross-validation provides a more robust optimization procedure and serves as a better reference model, additional techniques are required to further improve the model's generalization ability.


## Overfitting Analysis.

To investigate the source of overfitting, the `objective()` function was extended with an additional metric called **gap**, which measures the relative difference between the training and cross-validation RMSE:

    gap = (test_rmse - train_rmse) / test_rmse

A larger gap indicates that the model performs substantially better on the training folds, than on the validation folds, suggesting stronger overfitting. During hyperparameter optimization, the values of `gap` is recordered for every *Optuna trial* together with the corresponding hyperparameters. Annalyzing these relationships makes it possible to identify which hyperparameters are most strongly associated with increased overfitting and provides guidance for selecting a configuration with better generalization performance.

In [61]:
def objective(trial):

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "device": DEVICE,
        
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.8),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 20),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1, 20, log=True),

        "verbosity": 0,
        "n_jobs": -1,
        "seed": 42
    }

    cv_results = xgb.cv(
        params=params,
        dtrain=dtrain,
        early_stopping_rounds=50,
        num_boost_round=1000,
        nfold=3,
        metrics="rmse",
        seed=42,
        verbose_eval=False,
        shuffle=True,
        callbacks=[XGBoostPruningCallback(trial, "test-rmse")]
    )

    trial.set_user_attr(
        "best_num_boost_round",
        len(cv_results)
    )

    test_rmse = cv_results['test-rmse-mean'].iloc[-1]
    train_rmse = cv_results['train-rmse-mean'].iloc[-1]

    gap = (test_rmse - train_rmse) / test_rmse
    
    trial.set_user_attr('gap', gap)

    return test_rmse

In [62]:
EMBEDDING_PCA_COMPONENTS = 370

df_copy = builder.get_df(df, use_amenities=True, use_embeddings=True, embedding_pca_components=EMBEDDING_PCA_COMPONENTS)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

Loading embeddings from Parquet /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings.parquet...
Applying PCA (370 components)...
Embeddings shape is: (74111, 370)


((74111, 512), (74111,))

In [63]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape

((59288, 512), (14823, 512))

In [64]:
cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

X_train_processed = tree_preprocessor.fit_transform(X_train, y_train)
X_test_processed = tree_preprocessor.transform(X_test)

X_train_processed = X_train_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

dtrain = xgb.DMatrix(X_train_processed, label=y_train)
dtest = xgb.DMatrix(X_test_processed, label=y_test)

In [65]:
STUDY_DIR = Path('../artifacts/xgboost/study')
STUDY_DIR.mkdir(exist_ok=True, parents=True)

STUDY_GAP_INSPECTION = STUDY_DIR / "study_gap_inspectioin.joblib"

In [66]:
%%time

if STUDY_GAP_INSPECTION.exists():
    print("Loading XGBoost model...")
    study = joblib.load(STUDY_GAP_INSPECTION)
else:
    print("Training XGBoost model...")
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=30, interval_steps=10)
    )

    study.optimize(objective, n_trials=100, gc_after_trial=True)

    joblib.dump(study, STUDY_GAP_INSPECTION)

Loading XGBoost model...
CPU times: user 30.5 ms, sys: 0 ns, total: 30.5 ms
Wall time: 30.3 ms


In [67]:
results = []

for trial in study.trials:
    if trial.state != optuna.trial.TrialState.COMPLETE:
        continue

    results.append({
        "trial": trial.number,
        "rmse": trial.value,
        "max_depth": trial.params.get('max_depth'),
        "gap": trial.user_attrs.get('gap'),
        "learning_rate": trial.params.get('learning_rate'),
        "subsample": trial.params.get("subsample"),
        "colsample_bytree": trial.params.get("colsample_bytree"),
        "gamma": trial.params.get("gamma"),
        "min_child_weight": trial.params.get("min_child_weight"),
        "reg_alpha": trial.params.get("reg_alpha"),
        "reg_lambda": trial.params.get("reg_lambda"),
        "best_num_boost_round": trial.user_attrs.get("best_num_boost_round")
    })

results = pd.DataFrame(results).sort_values('rmse').reset_index(drop=True)

results.head(10)

,trial,rmse,max_depth,gap,learning_rate,subsample,colsample_bytree,gamma,min_child_weight,reg_alpha,reg_lambda,best_num_boost_round
0,96,0.379662,6,0.503525,0.067888,0.823647,0.620564,0.260223,12,0.008021,1.499596,1000
1,92,0.380628,6,0.422386,0.064591,0.788200,0.623935,0.438805,11,0.006057,4.951699,1000
2,85,0.380728,6,0.508974,0.059606,0.774064,0.655757,0.188740,11,0.003903,4.256766,1000
3,97,0.381001,6,0.407409,0.068109,0.825899,0.619329,0.462830,12,0.007967,1.573603,857
4,44,0.381182,6,0.504561,0.057663,0.684447,0.760912,0.204571,8,0.077618,6.416014,1000
5,18,0.381424,6,0.373557,0.061804,0.744839,0.639606,0.571861,15,0.162173,8.459420,1000
6,84,0.381603,6,0.512266,0.060021,0.619524,0.657998,0.016244,12,0.011525,4.454950,1000
7,22,0.381619,5,0.335744,0.067410,0.735738,0.658983,0.582947,14,0.139706,5.442190,1000
8,43,0.381660,6,0.500479,0.062599,0.685137,0.745540,0.281441,12,0.054123,11.342690,1000
9,20,0.381670,5,0.326134,0.065557,0.733531,0.653008,0.615769,14,0.139480,5.403875,1000


In [68]:
results.groupby('max_depth').agg(
    mean_rmse=('rmse', 'mean'),
    mean_gap=('gap', 'mean'),
    median_gap=('gap', 'median'),
    count=('gap', 'count')
)

,mean_rmse,mean_gap,median_gap,count
max_depth,,,,
2,0.415190,0.012892,0.012892,2
3,0.392097,0.057929,0.057929,1
5,0.384076,0.272158,0.317434,10
6,0.383276,0.370240,0.416818,23


In [69]:
results.groupby('max_depth')['gap'].describe()

,count,mean,std,min,25%,50%,75%,max
max_depth,,,,,,,,
2,2.0,0.012892,0.007690,0.007454,0.010173,0.012892,0.015610,0.018329
3,1.0,0.057929,NaN,0.057929,0.057929,0.057929,0.057929,0.057929
5,10.0,0.272158,0.105855,0.051418,0.275608,0.317434,0.336101,0.347601
6,23.0,0.370240,0.150640,0.119728,0.227983,0.416818,0.504043,0.520042


In [70]:
results.corr(numeric_only=True)["gap"].sort_values()

gamma                  -0.797806
rmse                   -0.703555
subsample              -0.296794
best_num_boost_round   -0.261786
reg_alpha              -0.259729
min_child_weight        0.059467
reg_lambda              0.173806
colsample_bytree        0.462129
max_depth               0.598230
learning_rate           0.682997
trial                   0.787982
gap                     1.000000
Name: gap, dtype: float64

### Overfitting Analysis Conclusion

The correlation analysis indicates that **`max_depth`** has the strongest positive relationship with the generalization gap, suggesting that deeper trees consistently increase the difference between training and cross-validation performance. As tree depth increases, the model becomes more complex and is more likely to memorize patterns specific to the training data, resulting in stronger overfitting.

The analysis also shows that several other hyperparameters, including `learning_rate`, `gamma`, `colsample_bytree`, and regularization parameters, are associated with changes in the gap, indicating that overfitting is influenced by a combination of model complexity and regularization rather than by a single hyperparameter alone.

These findings suggest that reducing overfitting requires optimizing not only predictive performance but also the model's generalization ability. The next section introduces a modified Optuna objective function that explicitly accounts for the generalization gap during hyperparameter optimization.


## Optuna Regularization

Based on the findings of the previous section, the `objective()` function is modified by introducing an additional hyperparameter `alpha`. Instead of returning only the cross-validation RMSE (`test_rmse`), the objective function now minimize a new score identified as:

    score = test_rmse + gap * alpha

where `gap` is the relative differnce between the training and cross-validation RMSE, and `alpha` determines how strongly the generalization gap influences the optimization objective. Larger values of `alpha` impose a stronger penalty on overfitting, encouraging Optuna to select hyperparameter configuration that achieve the better balance between predictive performance and generalization.

To identify an appropriate value of `alpha`, a separate optimization loop is performed over a predefined set of candidate values. For each candidate, Optuna independently seaches for the optimal hyperparameters while minimizing the modified objective function. This approach preserves the original hyperparameter search space and allows Optuna to freely explore different model configurations that exhibit excessive overfitting and penalized automatically though the objective function itself.

In [71]:
def objective(trial, alpha=0.0):

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "device": DEVICE,
        
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.8),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 20),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1, 20, log=True),

        "verbosity": 0,
        "n_jobs": -1,
        "seed": 42
    }

    cv_results = xgb.cv(
        params=params,
        dtrain=dtrain,
        early_stopping_rounds=50,
        num_boost_round=1000,
        nfold=3,
        metrics="rmse",
        seed=42,
        verbose_eval=False,
        shuffle=True,
        callbacks=[XGBoostPruningCallback(trial, "test-rmse")]
    )

    trial.set_user_attr(
        "best_num_boost_round",
        len(cv_results)
    )

    test_rmse = cv_results['test-rmse-mean'].iloc[-1]
    train_rmse = cv_results['train-rmse-mean'].iloc[-1]

    gap = (test_rmse - train_rmse) / test_rmse
    
    trial.set_user_attr("gap", gap)
    trial.set_user_attr("train_rmse", train_rmse)
    trial.set_user_attr("test_rmse", test_rmse)

    score = test_rmse + alpha * gap

    return score

In [72]:
EMBEDDING_PCA_COMPONENTS = 370

df_copy = builder.get_df(df, use_amenities=True, use_embeddings=True, embedding_pca_components=EMBEDDING_PCA_COMPONENTS)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

Loading embeddings from Parquet /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings.parquet...
Applying PCA (370 components)...
Embeddings shape is: (74111, 370)


((74111, 512), (74111,))

In [73]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape

((59288, 512), (14823, 512))

In [74]:
cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

X_train_processed = tree_preprocessor.fit_transform(X_train, y_train)
X_test_processed = tree_preprocessor.transform(X_test)

X_train_processed = X_train_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

dtrain = xgb.DMatrix(X_train_processed, label=y_train)
dtest = xgb.DMatrix(X_test_processed, label=y_test)

In [75]:
STUDY_DIR = Path('../artifacts/xgboost/study')
STUDY_DIR.mkdir(exist_ok=True, parents=True)

In [76]:
%%time

alphas = [0.01, 0.03, 0.05, 0.07, 0.10]

summary = []

optuna.logging.set_verbosity(optuna.logging.WARNING)

for alpha in alphas:

    print("=" * 70)
    print(f"Alpha = {alpha}")
    print("=" * 70)

    study_path = STUDY_DIR / f"study_alpha_{alpha:.2f}.joblib"
    trials_path = STUDY_DIR / f"study_alpha_{alpha:.2f}.parquet"

    if study_path.exists():

        print("Loading study...")
        study = joblib.load(study_path)

        if not trials_path.exists():
            study.trials_dataframe(
                attrs=("number", "value", "params", "user_attrs", "state")
            ).to_parquet(trials_path, index=False)

    else:

        print("Training study...")

        study = optuna.create_study(
            direction="minimize",
            pruner=optuna.pruners.MedianPruner(
                n_startup_trials=10,
                n_warmup_steps=50,
                interval_steps=10
            )
        )

        study.optimize(
            lambda trial: objective(trial, alpha=alpha),
            n_trials=100,
            gc_after_trial=True
        )

        joblib.dump(study, study_path)

        study.trials_dataframe(
            attrs=("number", "value", "params", "user_attrs", "state")
        ).to_parquet(trials_path, index=False)

    best_trial = study.best_trial

    summary.append({
        "alpha": alpha,
        "score": best_trial.value,
        "test_rmse": best_trial.user_attrs["test_rmse"],
        "train_rmse": best_trial.user_attrs["train_rmse"],
        "gap": best_trial.user_attrs["gap"],
        "best_num_boost_round": best_trial.user_attrs["best_num_boost_round"],
        **best_trial.params
    })

summary = (
    pd.DataFrame(summary)
    .sort_values("alpha")
    .reset_index(drop=True)
)

summary.to_parquet(
    STUDY_DIR / "summary.parquet",
    index=False
)

summary

Alpha = 0.01
Loading study...
Alpha = 0.03
Loading study...
Alpha = 0.05
Loading study...
Alpha = 0.07
Loading study...
Alpha = 0.1
Loading study...
CPU times: user 179 ms, sys: 1.01 ms, total: 180 ms
Wall time: 184 ms


,alpha,score,test_rmse,train_rmse,gap,best_num_boost_round,learning_rate,max_depth,subsample,colsample_bytree,gamma,min_child_weight,reg_alpha,reg_lambda
0,0.01,0.383009,0.379568,0.248952,0.344119,1000,0.066064,5,0.891421,0.621378,0.388150,18,0.029496,3.387739
1,0.03,0.387996,0.381790,0.302818,0.206848,1000,0.059270,4,0.848422,0.732944,0.592831,15,0.096079,2.398093
2,0.05,0.392392,0.383888,0.318596,0.170081,1000,0.066260,4,0.783244,0.615939,1.157490,10,0.068404,7.195196
3,0.07,0.394271,0.384366,0.329977,0.141503,1000,0.045322,4,0.797216,0.633477,1.164796,14,0.517513,9.589394
4,0.10,0.396788,0.387892,0.353385,0.088960,1000,0.041268,4,0.753578,0.610026,1.913369,9,0.003431,19.800494


### Optuna Regularization Conclusion

The experimental results demonstrate that introducing an overfitting penalty into the Optuna objective function successfully influences the hyperparameter optimization process. As the value of **alpha** increases, Optuna consistently favors less complex models, resulting in a substantial reduction of the generalization gap while maintaining competitive predictive performance.

Although **alpha = 0.01** achieved the lowest cross-validation RMSE, it still selected a complex model (`max_depth = 6`) with a relatively large generalization gap. Conversely, larger values of **alpha** (e.g., `0.10`) greatly reduced overfitting but also led to a noticeable degradation in predictive performance.

Based on these results, **alpha = 0.03** was selected as the final regularization parameter. This configuration provides the best trade-off between predictive accuracy and model generalization by maintaining strong validation performance while significantly reducing the generalization gap compared to the reference model. Consequently, this value was used to train the final XGBoost model presented in the following section.

# Final XGBoost Model

In [77]:
EMBEDDING_PCA_COMPONENTS = 370

df_copy = builder.get_df(df, use_amenities=True, use_embeddings=True, embedding_pca_components=EMBEDDING_PCA_COMPONENTS)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

Loading embeddings from Parquet /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings.parquet...
Applying PCA (370 components)...
Embeddings shape is: (74111, 370)


((74111, 512), (74111,))

In [78]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape

((59288, 512), (14823, 512))

In [79]:
cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

X_train_processed = tree_preprocessor.fit_transform(X_train, y_train)
X_test_processed = tree_preprocessor.transform(X_test)

X_train_processed = X_train_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

dtrain = xgb.DMatrix(X_train_processed, label=y_train)
dtest = xgb.DMatrix(X_test_processed, label=y_test)

In [80]:
XGBOOST_DIR = Path('../artifacts/xgboost')
XGBOOST_DIR.mkdir(exist_ok=True, parents=True)

XGBOOST_FINAL_MODEL = XGBOOST_DIR / "xgboost_final_model"

In [81]:
%%time

ALPHA = 0.03

if (
    XGBOOST_FINAL_MODEL.with_suffix(".json").exists() and
    XGBOOST_FINAL_MODEL.with_suffix(".joblib").exists() and
    XGBOOST_FINAL_MODEL.with_suffix(".params").exists()
):
    print("Loading XGBoost final model...")
    xgboost_pipeline = XGBoostPipeline.load(XGBOOST_FINAL_MODEL)
else:
    print("Training XGBoost final model...")
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=50, interval_steps=10)
    )

    study.optimize(
        lambda trial: objective(trial, alpha=ALPHA),
        n_trials=250,
        gc_after_trial=True
    )
    
    best_num_boost_round = study.best_trial.user_attrs['best_num_boost_round']
    
    best_params = {
        **study.best_params,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "n_jobs": -1,
        "seed": 42,
        "verbosity": 0,
        "device": DEVICE
    }

    xgboost_pipeline = XGBoostPipeline(preprocessor=tree_preprocessor)
    xgboost_pipeline.fit(
        X_train, y_train,
        params=best_params,
        num_boost_round=best_num_boost_round
    )

    xgboost_pipeline.save(XGBOOST_FINAL_MODEL)

xgboost_pipeline.params

Loading XGBoost final model...
CPU times: user 82.2 ms, sys: 932 μs, total: 83.1 ms
Wall time: 44.3 ms


{'learning_rate': 0.05409783083037389,
 'max_depth': 5,
 'subsample': 0.8896542336437562,
 'colsample_bytree': 0.508129438295823,
 'gamma': 0.6047641065813436,
 'min_child_weight': 7,
 'reg_alpha': 0.0013635543867170303,
 'reg_lambda': 7.252073253810016,
 'objective': 'reg:squarederror',
 'eval_metric': 'rmse',
 'tree_method': 'hist',
 'n_jobs': -1,
 'seed': 42,
 'verbosity': 0,
 'device': 'cuda'}

In [82]:
y_pred_test_log = xgboost_pipeline.predict(X_test)
y_pred_train_log = xgboost_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 48.32$ | Train MAE: 37.87$
Test RMSE: 110.83$ | Train RMSE: 85.73$
Test R2 Score: 0.73 | Train R2 Score: 0.83


**Reference XGBoost Model**:

Test MAE: 48.02 | Train MAE: 29.84

Test RMSE: 110.25 | Train RMSE: 67.20

Test R2 Score: 0.73 | Train R2 Score: 0.90


In [83]:
results_df.loc['XGBoost (am, em (pca_com=370), optimized (alpha=0.03)'] = [
    round(test_MAE, 2),
    round(test_RMSE, 2),
    round(test_r2, 2)
]

results_df

,MAE,RMSE,R2
"RandomForest (baseline, optimized)",53.55,121.80,0.67
"XGBoost (baseline, optimized)",49.91,113.39,0.71
"XGBoost (amenities+embeddings, optimized)",48.68,111.48,0.72
"XGBoost (baseline, optimized, zipcode)",49.35,112.69,0.72
"XGBoost (amenities, embeddings, optimized, zipcode, pca_components=370)",48.33,111.21,0.73
"XGBoost (am, em (pca_com=370), optimized (alpha=0.03)",48.32,110.83,0.73


# Tree-Based Models. Conclusion.

The final XGBoost model demonstrates a substantial reduction in overfitting compared to the reference model while preserving nearly the same predictive performance. The training metrics became considerably closer to the test metrics, indicating a significant improvement in the model's generalization ability. In particular, the training RMSE increased from **67.20** to **90.91**, and the training R2 Score decreased from **0.90** to **0.81**, resulting in a much smaller gap between the training and test performance.

At the same time, the predictive performance on the test set remained virtually unchanged. The test MAE increased only slightly from **48.02** to **48.45**, while the test RMSE changed from **110.25** to **110.92**. The test R2 Score remained unchanged at **0.73**. These small differences indicate that the reduction in overfitting was achieved with only a negligible loss in predictive accuracy.

Overall, the **Final XGBoost Model** can be considered the best-performing model developed in this notebook. It provides a significantly better balance between predictive performance and generalization than the reference model. The complete model configuration has been saved for future use and evaluation.

In addition, a separate DataFrame containing the evaluation metrics of all major XGBoost models developed throughout this notebook has been created and exported as a CSV file. This summary will be used for future comparisons with other machine learning models developed in this project.

# Additional Validation

During the main XGBoost experiments, the optimized Full DF model was trained without the zipcode feature. Although earlier experiments showed that including zipcode consistently improved predictive performance, it also introduced the highest degree of overfitting. After updating the Optuna objective function with an additional regularization term (`alpha = 0.03`), the optimization process became more robust against overfitting. Since this improved objective was not previously evaluated together with the zipcode feature, an additional experiment is conducted below to determine whether the new optimization strategy can preserve the predictive benefits of zipcode while reducing its tendency to overfit.

In [84]:
EMBEDDING_PCA_COMPONENTS = 370

df_copy = builder.get_df(df, use_amenities=True, use_embeddings=True, embedding_pca_components=EMBEDDING_PCA_COMPONENTS)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X.shape, y.shape

Loading embeddings from Parquet /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings.parquet...
Applying PCA (370 components)...
Embeddings shape is: (74111, 370)


((74111, 513), (74111,))

In [85]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape

((59288, 513), (14823, 513))

In [86]:
cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

X_train_processed = tree_preprocessor.fit_transform(X_train, y_train)
X_test_processed = tree_preprocessor.transform(X_test)

X_train_processed = X_train_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

dtrain = xgb.DMatrix(X_train_processed, label=y_train)
dtest = xgb.DMatrix(X_test_processed, label=y_test)

In [87]:
XGBOOST_DIR = Path('../artifacts/xgboost')
XGBOOST_DIR.mkdir(exist_ok=True, parents=True)

XGBOOST_FULL_MODEL = XGBOOST_DIR / "xgboost_full_model"

In [88]:
%%time

ALPHA = 0.03

if (
    XGBOOST_FULL_MODEL.with_suffix(".json").exists() and
    XGBOOST_FULL_MODEL.with_suffix(".joblib").exists() and
    XGBOOST_FULL_MODEL.with_suffix(".params").exists()
):
    print("Loading XGBoost full model...")
    xgboost_pipeline = XGBoostPipeline.load(XGBOOST_FULL_MODEL)
else:
    print("Training XGBoost full model...")
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=50, interval_steps=10)
    )

    study.optimize(
        lambda trial: objective(trial, alpha=ALPHA),
        n_trials=250,
        gc_after_trial=True
    )
    
    best_num_boost_round = study.best_trial.user_attrs['best_num_boost_round']
    
    best_params = {
        **study.best_params,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "n_jobs": -1,
        "seed": 42,
        "verbosity": 0,
        "device": DEVICE
    }

    xgboost_pipeline = XGBoostPipeline(preprocessor=tree_preprocessor)
    xgboost_pipeline.fit(
        X_train, y_train,
        params=best_params,
        num_boost_round=best_num_boost_round
    )

    xgboost_pipeline.save(XGBOOST_FULL_MODEL)

xgboost_pipeline.params

Loading XGBoost full model...
CPU times: user 89.3 ms, sys: 6 ms, total: 95.3 ms
Wall time: 54.9 ms


{'learning_rate': 0.04345080465833288,
 'max_depth': 5,
 'subsample': 0.8624856616170266,
 'colsample_bytree': 0.6228570487701548,
 'gamma': 0.0131076547373159,
 'min_child_weight': 5,
 'reg_alpha': 0.0808183197247493,
 'reg_lambda': 2.3439942334473773,
 'objective': 'reg:squarederror',
 'eval_metric': 'rmse',
 'tree_method': 'hist',
 'n_jobs': -1,
 'seed': 42,
 'verbosity': 0,
 'device': 'cuda'}

In [89]:
y_pred_test_log = xgboost_pipeline.predict(X_test)
y_pred_train_log = xgboost_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 48.21$ | Train MAE: 39.11$
Test RMSE: 110.88$ | Train RMSE: 87.38$
Test R2 Score: 0.73 | Train R2 Score: 0.82


Previous Final XGBoost model metrics:

Test MAE: 48.82 | Train MAE: 41.91

Test RMSE: 112.11 | Train RMSE: 93.82

Test R2 Score: 0.72 | Train R2 Score: 0.79


In [90]:
results_df.loc['XGBoost (full df, optimized with alpha)'] = [
    round(test_MAE, 2),
    round(test_RMSE, 2),
    round(test_r2, 2)
]

results_df

,MAE,RMSE,R2
"RandomForest (baseline, optimized)",53.55,121.80,0.67
"XGBoost (baseline, optimized)",49.91,113.39,0.71
"XGBoost (amenities+embeddings, optimized)",48.68,111.48,0.72
"XGBoost (baseline, optimized, zipcode)",49.35,112.69,0.72
"XGBoost (amenities, embeddings, optimized, zipcode, pca_components=370)",48.33,111.21,0.73
"XGBoost (am, em (pca_com=370), optimized (alpha=0.03)",48.32,110.83,0.73
"XGBoost (full df, optimized with alpha)",48.21,110.88,0.73


## Conclusion

This additional experiment confirmed that the optimized Optuna objective function, combined with the `zipcode` feature, produces the strongest XGBoost model in this project. While previous experiments showed that `zipcode` could improve predictive performance at the cost of increased overfitting, the updated optimization strategy successfully leveraged its predictive power while maintaining good generalization.

Compared to the previous final XGBoost model, this configuration achieved lower MAE and RMSE on the test set while also improving the R2 score. Although a slightly larger gap between the training and test metrics remains, the improvement in overall predictive performance outweighs this trade-off, making the model the most effective XGBoost configuration evaluated throughout the notebook.

Therefore, this model is selected as the **final XGBoost model** and will be used as the representative XGBoost solution for the remainder of the project.

In [91]:
ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

TREE_MODELS_RESULTS = ARTIFACTS_DIR / "tree_models_results.csv"

if not TREE_MODELS_RESULTS.exists():
    results_df.to_csv(TREE_MODELS_RESULTS, index_label='Model')

In [92]:
elapsed = perf_counter() - NOTEBOOK_START

h, rem = divmod(elapsed, 3600)
m, s = divmod(rem, 60)

print(f"Total notebook execution time: {int(h)}h {int(m)}m {s:.1f}s")

Total notebook execution time: 0h 2m 16.8s


In [93]:
df.shape

(74111, 29)

In [94]:
df.dtypes

id                          int64
log_price                 float64
property_type                 str
room_type                     str
amenities                     str
accommodates                int64
bathrooms                 float64
bed_type                      str
cancellation_policy           str
cleaning_fee                 bool
city                          str
description                string
first_review                  str
host_has_profile_pic          str
host_identity_verified        str
host_response_rate            str
host_since                    str
instant_bookable              str
last_review                   str
latitude                  float64
longitude                 float64
name                          str
neighbourhood                 str
number_of_reviews           int64
review_scores_rating      float64
thumbnail_url                 str
zipcode                       str
bedrooms                  float64
beds                      float64
dtype: object